In [18]:
import numpy as np 
from collections import Counter

In [ ]:
class NGramModel:

    def __init__(self,step_size):

        self.word_frequency = Counter()
        self.total_words = 0 
        self.step_size = step_size 
        self.probabilities = {}
        self.n_gram_frquency={}

    def train(self,X):

        self.probabilities = {}


        tokens = X.split(" ")
        n = len(tokens)

        self.word_frequency = Counter(tokens)
        self.total_words = n


        self.n_gram_frquency = Counter()

        for i in range(n-self.step_size+1):

            t_gram = tuple(tokens[i:i+self.step_size])
            self.n_gram_frquency[t_gram]+=1


        context_freq = Counter()

        for n_gram , count in self.n_gram_frquency.items():

            context_freq[n_gram[:-1]]+=count 


        for ngram,count in self.n_gram_frquency.items():

            previous_given = ngram[:-1]

            denominator = context_freq[ngram[:-1]] + len(self.word_frequency)
            numerator = count + 1 


            self.probabilities[ngram] = numerator/denominator

    def predict_next(self, text):
        tokens = text.split()

        context = tuple(tokens[-(self.step_size - 1):])


        context_count  = sum(
            count for ngram, count in self.n_gram_frquency.items()
            if ngram[:-1] == context
        )

        candidates = {}

        for word,freq in self.word_frequency.items():
            n_gram = context + (word,)
            count = self.n_gram_frquency.get(n_gram,0)


            if context_count > 0:

                probability = (count + 1) / (context_count + len(self.word_frequency))

            else:

                probability = (self.word_frequency[word]+1)/(self.total_words  + len(self.word_frequency))

            candidates[word] = probability

        if not candidates:
            return None

        # Return the most probable next token
        return max(candidates, key=candidates.get)

In [25]:
X = "I love Python I love Java I love Python"

model = NGramModel(step_size=2)
model.train(X)

for text in ["I love", "Do you"]:
    print(text, "->", model.predict_next(text))

I love -> Python
Do you -> I
